# 01 Baseline Replay
Thin notebook: orchestration only. Business logic lives in src/.

In [1]:
import os
import sys
import importlib
from IPython.display import display

cwd = os.getcwd()
candidates = [cwd, os.path.dirname(cwd)]
REPO = next((p for p in candidates if os.path.exists(os.path.join(p, 'src'))), cwd)
SRC = os.path.join(REPO, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import margin_analysis
importlib.reload(margin_analysis)
import charts
importlib.reload(charts)
import replay_analyzer
importlib.reload(replay_analyzer)
from replay_analyzer import default_analyzer_for_repo

In [2]:
analyzer = default_analyzer_for_repo(REPO)
baseline = analyzer.replay_folder(os.path.join(REPO, 'data', 'baseline'), export_dir=os.path.join(REPO, 'output'))

print('Detected Baseline Configuration')
display(baseline.configuration())

print('Summary Metrics')
display(baseline.summary())

print('Per Pair Statistics')
display(baseline.pair_table())

No XLSX found for EURGBP, skipping
No XLSX found for GBPUSD, skipping
No XLSX found for USDCHF, skipping


Detected Baseline Configuration


,Pair,Risk,TP,Grid,Max Trades,Trades,Initial Balance,Median Lot
0,CADJPY,1.4,15,60,3,309,5000.0,0.09
1,EURUSD,2.8,15,54,4,359,5000.0,0.28
2,NZDCHF,1.8,10,31,5,315,5000.0,0.12


Summary Metrics


,Initial Balance,Final Balance,Profit,Average Monthly Growth,CAGR,Max Floating Drawdown %,Peak-to-Trough Equity Drawdown %,Minimum Margin Level,Minimum Free Margin,Profit Factor,Recovery Factor,Total Trades,Winning Trades,Win Rate
0,5000.0,24336.9423,19336.9423,3.421279,50.9384,-43.310612,38.259045,109.1363,832.8155,2.09122,2.210622,1966,834,42.42116


Per Pair Statistics


,Pair,Risk,Baseline Risk,Take Profit,Grid,Max Trades,Trades,Scaled PnL
0,CADJPY,1.4,1.4,15,60,3,309,3675.3342
1,EURUSD,2.8,2.8,15,54,4,359,11116.3564
2,NZDCHF,1.8,1.8,10,31,5,315,4545.2517


In [3]:
import pandas as pd
import margin_analysis
import charts

ma = baseline.margin_analysis()
events = ma.events_df()
threshold_summary = ma.threshold_summary_df(events)

print('Portfolio Margin Summary')
display(pd.DataFrame([ma.portfolio_summary_dict(events)]).T.rename(columns={0: 'value'}))

print('\nThreshold Summary (Canonical Events)')
display(threshold_summary)

print('\nPer-Pair Contribution Summary')
display(ma.pair_summary_df(events))

if not events.empty:
    print(f'\nCanonical Margin Events ({len(events)} episodes) — worst 10 by margin level')
    display(events.nsmallest(10, 'min_margin_level')[
        ['event_id', 'start_time', 'end_time', 'duration_minutes',
         'min_margin_level', 'worst_threshold_crossed', 'thresholds_crossed',
         'max_used_margin', 'largest_used_margin_contributor',
         'largest_used_margin_contribution_pct']
    ])

    print('\nBasket Composition at 3 Worst Events')
    display(ma.event_baskets_df(events.nsmallest(3, 'min_margin_level')))

Portfolio Margin Summary


,value
min_margin_level,109.14
avg_margin_level,609.70
median_margin_level,412.86
p95_used_margin,9616.88
max_used_margin,14254.15
max_free_margin,29426.77
min_free_margin,832.82
total_minutes_below_300,587037.00
event_count_below_300,172.00
longest_event_minutes_below_300,85995.00



Threshold Summary (Canonical Events)


,threshold,event_count,total_minutes_below,longest_event_minutes_below,worst_margin_level
0,300.0,172,587037,85995,109.14
1,200.0,12,454801,85995,109.14
2,150.0,3,186422,85995,109.14
3,100.0,0,0,0,NaN



Per-Pair Contribution Summary


,pair,avg_used_margin,max_used_margin,avg_used_margin_contribution_pct,peak_used_margin_contribution_pct,avg_floating_pnl,worst_floating_pnl,avg_open_positions,max_open_positions,margin_events_participated,margin_events_as_largest_contributor
0,CADJPY,973.06,4629.36,28.22,100.0,-406.44,-3111.92,1.38,3,134,45
1,EURUSD,1461.40,7946.00,33.94,100.0,-610.56,-7808.77,1.46,4,160,102
2,NZDCHF,1030.78,2891.20,36.25,100.0,-359.99,-3215.52,2.20,5,171,25



Canonical Margin Events (172 episodes) — worst 10 by margin level


,event_id,start_time,end_time,duration_minutes,min_margin_level,worst_threshold_crossed,thresholds_crossed,max_used_margin,largest_used_margin_contributor,largest_used_margin_contribution_pct
128,evt_1745283600_129,2025-04-22 01:00:00,2025-06-20 18:15:00,85995,109.14,150.0,"300,200,150",9231.99,EURUSD,45.94
78,evt_1715072400_79,2024-05-07 09:00:00,2024-06-10 06:45:00,48825,125.72,150.0,"300,200,150",4886.23,EURUSD,37.96
63,evt_1708011000_64,2024-02-15 15:30:00,2024-03-22 11:32:14,51602,137.50,150.0,"300,200,150",5133.24,EURUSD,47.21
156,evt_1772737200_157,2026-03-05 19:00:00,2026-03-19 21:30:00,20310,153.24,200.0,"300,200",11702.10,EURUSD,54.25
171,evt_1791559800_172,2026-10-09 15:30:00,2026-11-05 23:45:00,39375,162.91,200.0,"300,200",14254.15,EURUSD,58.19
79,evt_1720090800_80,2024-07-04 11:00:00,2024-08-01 11:00:00,40320,171.08,200.0,"300,200",4584.17,EURUSD,41.31
169,evt_1774474200_170,2026-03-25 21:30:00,2026-04-08 08:45:00,19395,182.83,200.0,"300,200",10476.73,EURUSD,61.42
170,evt_1786539600_171,2026-08-12 13:00:00,2026-09-16 21:00:00,50880,186.41,200.0,"300,200",12357.54,EURUSD,64.30
153,evt_1765947600_154,2025-12-17 05:00:00,2026-01-08 15:15:00,32295,188.23,200.0,"300,200",9455.57,EURUSD,61.19
135,evt_1750867200_136,2025-06-25 16:00:00,2025-07-02 14:45:00,10005,193.92,200.0,"300,200",7359.93,NZDCHF,37.81



Basket Composition at 3 Worst Events


,event_id,worst_threshold_crossed,thresholds_crossed,min_margin_time,pair,open_positions,used_margin,floating_pnl,used_margin_contribution_pct,portfolio_used_margin,portfolio_margin_level
0,evt_1745283600_129,150.0,"300,200,150",2025-05-16 19:15:00,CADJPY,3,2251.28,-1676.55,24.70,9115.49,109.14
1,evt_1745283600_129,150.0,"300,200,150",2025-05-16 19:15:00,EURUSD,4,4187.74,-2879.33,45.94,9115.49,109.14
2,evt_1745283600_129,150.0,"300,200,150",2025-05-16 19:15:00,NZDCHF,5,2676.47,-1966.25,29.36,9115.49,109.14
3,evt_1715072400_79,150.0,"300,200,150",2024-05-28 06:30:00,CADJPY,3,1358.65,-1629.92,27.89,4871.75,125.72
4,evt_1715072400_79,150.0,"300,200,150",2024-05-28 06:30:00,EURUSD,3,1849.26,-913.46,37.96,4871.75,125.72
5,evt_1715072400_79,150.0,"300,200,150",2024-05-28 06:30:00,NZDCHF,5,1663.84,-1339.83,34.15,4871.75,125.72
6,evt_1708011000_64,150.0,"300,200,150",2024-03-21 13:00:00,CADJPY,3,1371.91,-922.02,26.95,5090.22,137.50
7,evt_1708011000_64,150.0,"300,200,150",2024-03-21 13:00:00,EURUSD,4,2402.86,-1051.89,47.21,5090.22,137.50
8,evt_1708011000_64,150.0,"300,200,150",2024-03-21 13:00:00,NZDCHF,4,1315.45,-689.18,25.84,5090.22,137.50


## Margin Analysis

In [4]:
fig_equity = baseline.plot_equity()
fig_pair_balance = baseline.plot_pair_contributions()
#fig_pair_floating = baseline.plot_pair_floating()
fig_pair_drawdown = baseline.plot_pair_drawdown()
fig_margin = baseline.plot_margin()
fig_monthly_growth = baseline.plot_monthly_growth() 

fig_equity.show()
fig_pair_balance.show()
#fig_pair_floating.show()
fig_pair_drawdown.show()
fig_margin.show()
fig_monthly_growth.show()

## Swap Reports
Focused swap diagnostics for the baseline replay.

In [5]:
import os
import pandas as pd
from IPython.display import display

# Clean up stale swap_position_summary.csv from previous runs with different pairs
pos_path = os.path.join(REPO, 'output', 'swap_position_summary.csv')
if os.path.exists(pos_path):
    os.remove(pos_path)
    print(f'Removed stale {pos_path} to prevent displaying outdated position data.')

replay_df = baseline.replay_table().copy()
replay_df['time_dt'] = pd.to_datetime(replay_df['time'], format='%Y.%m.%d %H:%M:%S')
swap_df = replay_df[replay_df['EventType'] == 'swap'].copy()

if swap_df.empty:
    print('No swap rows were found in the current baseline replay output.')
else:
    swap_df['post_date'] = swap_df['time_dt'].dt.date
    swap_df['month'] = swap_df['time_dt'].dt.to_period('M').astype(str)

    total_swap = float(swap_df['ModeledSwap'].sum())
    swap_events = int(len(swap_df))
    avg_swap_per_event = float(total_swap / swap_events)
    avg_daily_swap = float(swap_df.groupby('post_date')['ModeledSwap'].sum().mean())

    deal_df = replay_df[replay_df['EventType'].astype(str).str.startswith('deal_')].copy()
    realized_profit = float(deal_df['scaled_net_profit'].sum())
    swap_pct_realized = (total_swap / realized_profit * 100.0) if realized_profit != 0 else float('nan')

    print('Swap summary')
    display(pd.DataFrame([
        {'Metric': 'Total modeled swap', 'Value': total_swap},
        {'Metric': 'Swap events', 'Value': swap_events},
        {'Metric': 'Average swap per event', 'Value': avg_swap_per_event},
        {'Metric': 'Average daily swap', 'Value': avg_daily_swap},
        {'Metric': 'Swap as % of realized profit', 'Value': swap_pct_realized},
    ]))

    print('Swap by pair')
    display(swap_df.groupby('pair', as_index=False)['ModeledSwap'].sum().sort_values('ModeledSwap'))

    print('Swap by direction')
    display(swap_df.groupby('direction', as_index=False)['ModeledSwap'].sum().sort_values('ModeledSwap'))

    print('Monthly swap')
    display(swap_df.groupby('month', as_index=False)['ModeledSwap'].sum().sort_values('month'))

    print('Top 20 positions by total swap')
    if os.path.exists(pos_path):
        pos_df = pd.read_csv(pos_path)
        display(pos_df.sort_values('Total Swap').head(20))
    else:
        print('swap_position_summary.csv not found in output/. Run swap_audit_report.py to generate it.')

    print('Cumulative swap over time (last 20 days)')
    daily = swap_df.groupby('post_date', as_index=False)['ModeledSwap'].sum().sort_values('post_date')
    daily['CumulativeSwap'] = daily['ModeledSwap'].cumsum()
    display(daily.tail(20))


Swap summary


,Metric,Value
0,Total modeled swap,-9049.100000
1,Swap events,7024.000000
2,Average swap per event,-1.288312
3,Average daily swap,-6.528932
4,Swap as % of realized profit,-31.878694


Swap by pair


,pair,ModeledSwap
1,EURUSD,-4372.87
2,NZDCHF,-3367.65
0,CADJPY,-1308.58


Swap by direction


,direction,ModeledSwap
1,sell,-5004.75
0,buy,-4044.35


Monthly swap


,month,ModeledSwap
0,2023-01,-2.00
1,2023-02,-117.62
2,2023-03,-106.09
3,2023-04,13.12
4,2023-05,-107.84
5,2023-06,-152.35
6,2023-07,-27.58
7,2023-08,-21.58
8,2023-09,-182.37
9,2023-10,-370.65


Top 20 positions by total swap
swap_position_summary.csv not found in output/. Run swap_audit_report.py to generate it.
Cumulative swap over time (last 20 days)


,post_date,ModeledSwap,CumulativeSwap
1366,2026-10-17,-42.15,-8148.43
1367,2026-10-18,-42.15,-8190.58
1368,2026-10-19,-42.15,-8232.73
1369,2026-10-20,-42.15,-8274.88
1370,2026-10-21,-48.65,-8323.53
1371,2026-10-22,-51.25,-8374.78
1372,2026-10-23,-44.77,-8419.55
1373,2026-10-24,-51.25,-8470.80
1374,2026-10-25,-51.25,-8522.05
1375,2026-10-26,-51.25,-8573.30
